In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdb
from IPython.display import display
import os

In [23]:
import ee
import geemap
service_account = ' boba-account@boba-430314.iam.gserviceaccount.com '
credentials = ee.ServiceAccountCredentials(service_account,'C:/Users/arj26323/OneDrive - University of Georgia/Documents/GEE AUTH/boba-430314-a19be859bfca.json')

ee.Initialize(credentials)


# ee.Authenticate()
# ee.Initialize()



print(geemap.__version__)

0.34.1


In [24]:
# df = pd.read_csv('C:/Users/arj26323/Documents/Data/GA_conservation/all_active_years_installations.csv')
df = pd.read_csv('C:/Users/arj26323/Documents/Data/GA_conservation/actives.csv')
path = 'C:/Users/arj26323/Documents/Data/GA_conservation/actives.csv'
# df

fc_all = geemap.csv_to_ee(
    path, 
    latitude="Latitude", 
    longitude="Longitude"
)


In [25]:
Map = geemap.Map(center=[31.539096,-81.422318], zoom=10)

Map.add_layer(fc_all, {}, 'Restoration sites')

Map

Map(center=[31.539096, -81.422318], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

In [16]:
def maskL8sr(image):
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)
    # Apply the scaling factors to the appropriate bands.
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    # Replace the original bands with the scaled ones and apply the masks.
    return image.addBands(opticalBands, None, True) \
    .addBands(thermalBands, None, True) \
    .updateMask(qaMask) \
    .updateMask(saturationMask)

#12/08/22 - adding CALIBRATED FLATS
def addFLATSL7(image):
    flats = ee.Image(0).expression(
        '1/(1+2.718281828459045**-(1.51 + 12.5*(RED-SWIR)/(RED+SWIR) - 41.2*(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)))', {
            'SWIR': image.select('SR_B5'),
            'NIR': image.select('SR_B4'),
            'RED': image.select('SR_B3'),
            'BLUE': image.select('SR_B1')
        })
    
    return image.addBands(flats.rename('flats'))

def addFLATSL5(image):
    flats = ee.Image(0).expression(
        '1/(1+2.718281828459045**-(1.51 + 12.5*(0.972*(RED-SWIR)/(RED+SWIR)-0.008) - 41.2*(0.991*(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)-0.0014)))', {
            'SWIR': image.select('SR_B5'),
            'NIR': image.select('SR_B4'),
            'RED': image.select('SR_B3'),
            'BLUE': image.select('SR_B1')
        })
    
    return image.addBands(flats.rename('flats'))


def addFLATSL8(image):
    flats = ee.Image(0).expression(
        '1/(1+2.718281828459045**-(1.51 + 12.5*(0.841*(RED-SWIR)/(RED+SWIR) - 0.019) - 41.2*(0.771*(NIR-RED)/(NIR+6*RED-7.5*BLUE+1) + 0.011)))', {
            'SWIR': image.select('SR_B6'),
            'NIR': image.select('SR_B5'),
            'RED': image.select('SR_B4'),
            'BLUE': image.select('SR_B2')
        })
    
    return image.addBands(flats.rename('flats'))

def addFLATSL9(image):
    flats = ee.Image(0).expression(
        '1/(1+2.718281828459045**-(1.51 + 12.5*(1.225*(RED-SWIR)/(RED+SWIR) + 0.096) - 41.2*(1.038* (NIR-RED)/(NIR+6*RED-7.5*BLUE+1) - 0.004)))', {
            'SWIR': image.select('SR_B6'),
            'NIR': image.select('SR_B5'),
            'RED': image.select('SR_B4'),
            'BLUE': image.select('SR_B2')
        })
    
    return image.addBands(flats.rename('flats'))

##MASKING FLATS
def maskFLATS(image):
    mask1 = image.select('flats').lte(0.2)
    return image.updateMask(mask1)

#EVI
def addevi_l57(image):
    evi = ee.Image(0).expression(
        '(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)', {
            'RED': image.select('SR_B3'),
            'NIR': image.select('SR_B4'),
            'BLUE': image.select('SR_B1'),

        })
    
    return image.addBands(evi.rename('evi'))

def addevi_l89(image):
    evi = ee.Image(0).expression(
        '(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)', {
            'RED': image.select('SR_B4'),
            'NIR': image.select('SR_B5'),
            'BLUE': image.select('SR_B2'),

        })
    
    return image.addBands(evi.rename('evi'))

In [17]:
##Pixel extraction functions - addDate for dateless images/collections -- ADD 3x3 WINDOW!
def addDate(image):
    img_date = ee.Date(image.date())
    img_date = ee.Number.parse(img_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(img_date).rename('imagedate').toInt())


# def rasterExtraction(image, fc_subset):
#     return image.sampleRegions(
#         collection=fc_subset,  # Use the subset of locations
#         scale=30,
#         tileScale=8  # Adjust tile scale for performance
#     ).map(lambda f: f.set('date', image.date().format('YYYY-MM-dd')))



def rasterExtraction(image, fc_subset):

    image_3x3 = image.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=ee.Kernel.square(
            radius=1,
            units='pixels'
        )
    )

    return (
        image_3x3.sampleRegions(
            collection=fc_subset,
            scale=30,
            tileScale=8
        )
        .map(lambda f: f.set('date', image.date().format('YYYY-MM-dd')))
    )

In [18]:
#Datasets - NOT MASKING FLATS (try again masking later)
ls5_collect = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(fc_all).map(maskL8sr).map(addFLATSL5).map(addevi_l57)
ls7_collect = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterBounds(fc_all).map(maskL8sr).map(addFLATSL7).map(addevi_l57)
ls8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(fc_all).map(maskL8sr).map(addFLATSL8).map(addevi_l89)
ls9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(fc_all).map(maskL8sr).map(addFLATSL8).map(addevi_l89)

#Merge
ultra_col = ls8_collect.merge(ls5_collect).merge(ls7_collect).merge(ls9_collect)

In [19]:
fc = fc_all

ultra_col = ultra_col.map(lambda img: img.select(['evi', 'flats']).toFloat())

# 2. Function to create monthly composites
def get_monthly_composites(start_year, end_year):
    months = ee.List.sequence(0, (end_year - start_year) * 12 - 1)
    
    def create_month(m):
        start = ee.Date.fromYMD(start_year, 1, 1).advance(m, 'month')
        end = start.advance(1, 'month')
        # Mean reduction for the month
        img = ultra_col.filterDate(start, end).mean()
        return ee.Algorithms.If(
            img.bandNames().size().gt(0), 
            img.set('system:time_start', start.millis()).set('date', start.format('YYYY-MM')),
            None # This effectively drops the image from the collection
    )
    
    return ee.ImageCollection(months.map(create_month))

# 3. Create the monthly collection
monthly_col = get_monthly_composites(1984, 2023)

# 4. Define the Extraction function
def extract_monthly_data(img):
    # Apply 3x3 window mean
    img_3x3 = img.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=ee.Kernel.square(radius=1, units='pixels')
    )
    # Extract values to points
    return img_3x3.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.first(),
        scale=30
    ).map(lambda f: f.set('date', img.get('date')))

# 5. Map the function and flatten into one FeatureCollection
# This creates a collection of features, one for each site for each month
all_data = monthly_col.map(extract_monthly_data).flatten()

In [20]:
monthly_col

In [21]:
import time

# 6. Export to Google Drive to bypass memory/element limits
task = ee.batch.Export.table.toDrive(
    collection=all_data,
    folder='GEE_Exports',  
    description='Landsat_40Year_TimeSeries',
    fileFormat='CSV'
)

task.start()
print("Export task started! Check your GEE Tasks tab.")

# 7. Monitor task status
while task.active():
    print(f"Task status: {task.status()['state']}...")
    time.sleep(60)
print("Export complete.")

Export task started! Check your GEE Tasks tab.
Task status: READY...
Export complete.


In [11]:
#Fin

[]
